In [1]:
%pip -q install -U deepl datasets evaluate sacrebleu


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import random
from typing import Optional, Dict, Tuple
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import deepl
import evaluate
from datasets import Dataset

TMX_ROOT = r"J:\\FINAL PROJECT"
TMX_EN_ES = os.path.join(TMX_ROOT, "data", "en-es.tmx")
TMX_EN_PT = os.path.join(TMX_ROOT, "data", "en-pt.tmx")

PROJECT_DIR = "./deepl_scielo"
os.makedirs(PROJECT_DIR, exist_ok=True)

SEED = 42
MAX_TRAIN = 800
MAX_VALID = 200
SPLIT_RATIO = 0.90

PAIR = "en-es"   # change to "en-es" for English->Spanish

random.seed(SEED)
np.random.seed(SEED)


In [3]:
# Option 1: set it in your notebook for this session
os.environ["DEEPL_API_KEY"] = "56f67712-73e4-4e7f-841c-b922ffcb30ed:fx"

# Option 2: if already set in system environment, keep this only
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")
assert DEEPL_API_KEY, "DEEPL_API_KEY is not set"

translator = deepl.Translator(DEEPL_API_KEY)
print("DeepL client initialized.")


DeepL client initialized.


In [4]:
_LANG_ALIASES = {
    "en": {"en", "eng", "en-us", "en-gb"},
    "es": {"es", "spa", "es-es", "es-la", "es-mx"},
    "pt": {"pt", "por", "pt-pt", "pt-br"},
}

def _norm_lang(code: str) -> str:
    c = (code or "").lower()
    for k, aliases in _LANG_ALIASES.items():
        if c in aliases:
            return k
    m = re.match(r"^([a-z]{2})[-_][a-z]{2}$", c)
    if m and m.group(1) in _LANG_ALIASES:
        return m.group(1)
    return c

def _extract_pair_from_tu(tu_elem, src_key, tgt_key):
    texts = {}
    for tuv in tu_elem.findall(".//tuv"):
        lang = _norm_lang(
            tuv.attrib.get("{http://www.w3.org/XML/1998/namespace}lang")
            or tuv.attrib.get("lang") or ""
        )
        seg = tuv.find(".//seg")
        if seg is not None and seg.text is not None:
            texts[lang] = seg.text.strip()
    if src_key in texts and tgt_key in texts:
        return texts[src_key], texts[tgt_key]
    return None

def read_tmx_pairs(path: str, src_key: str, tgt_key: str, max_samples: Optional[int] = None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"TMX not found: {path}")

    src, tgt = [], []
    for event, elem in ET.iterparse(path, events=("end",)):
        if elem.tag.lower().endswith("tu"):
            pair = _extract_pair_from_tu(elem, src_key, tgt_key)
            if pair:
                s, t = pair
                if s and t:
                    src.append(s)
                    tgt.append(t)
                    if max_samples and len(src) >= max_samples:
                        break
            elem.clear()

    return src, tgt


In [5]:
TAG = re.compile(r"</?[^>]+>")
WS = re.compile(r"\s+")

def _norm_text(x: str) -> str:
    x = x or ""
    x = TAG.sub(" ", x)
    x = WS.sub(" ", x)
    return x.strip()

def global_clean_and_dedup(src_all, tgt_all, min_ratio=0.5, max_ratio=1.8, min_len=5):
    rows = []
    for s, t in zip(src_all, tgt_all):
        s2, t2 = _norm_text(s), _norm_text(t)
        if not s2 or not t2:
            continue

        sw, tw = s2.split(), t2.split()
        if len(sw) < min_len or len(tw) < min_len:
            continue

        r = len(tw) / max(1, len(sw))
        if r < min_ratio or r > max_ratio:
            continue

        rows.append((s2, t2))

    seen = set()
    uniq = []
    for s2, t2 in rows:
        if s2 in seen:
            continue
        seen.add(s2)
        uniq.append((s2, t2))

    return [s for s, _ in uniq], [t for _, t in uniq]

def split_to_datasets(src_clean, tgt_clean, split_ratio=0.9, seed=42, max_train=None, max_valid=None):
    idxs = list(range(len(src_clean)))
    rng = random.Random(seed)
    rng.shuffle(idxs)

    cut = int(len(idxs) * split_ratio)
    train_idx, valid_idx = idxs[:cut], idxs[cut:]

    if max_train is not None:
        train_idx = train_idx[:max_train]
    if max_valid is not None:
        valid_idx = valid_idx[:max_valid]

    def build(idxs):
        return Dataset.from_dict({
            "src_text": [src_clean[i] for i in idxs],
            "tgt_text": [tgt_clean[i] for i in idxs],
        })

    return build(train_idx), build(valid_idx)


In [6]:
print("EN-ES TMX:", os.path.abspath(TMX_EN_ES))
src_es_all, tgt_es_all = read_tmx_pairs(TMX_EN_ES, "en", "es")
src_es_clean, tgt_es_clean = global_clean_and_dedup(src_es_all, tgt_es_all)
train_es, valid_es = split_to_datasets(
    src_es_clean, tgt_es_clean,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    max_train=MAX_TRAIN,
    max_valid=MAX_VALID
)

print("EN-PT TMX:", os.path.abspath(TMX_EN_PT))
src_pt_all, tgt_pt_all = read_tmx_pairs(TMX_EN_PT, "en", "pt")
src_pt_clean, tgt_pt_clean = global_clean_and_dedup(src_pt_all, tgt_pt_all)
train_pt, valid_pt = split_to_datasets(
    src_pt_clean, tgt_pt_clean,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    max_train=MAX_TRAIN,
    max_valid=MAX_VALID
)

print("EN→ES:", len(train_es), len(valid_es))
print("EN→PT:", len(train_pt), len(valid_pt))


EN-ES TMX: J:\FINAL PROJECT\data\en-es.tmx
EN-PT TMX: J:\FINAL PROJECT\data\en-pt.tmx
EN→ES: 800 200
EN→PT: 800 200


In [7]:
if PAIR == "en-es":
    train_ds, valid_ds = train_es, valid_es
    LANG = "es"
    LANG_NAME = "Spanish"
    DEEPL_TARGET = "ES"
elif PAIR == "en-pt":
    train_ds, valid_ds = train_pt, valid_pt
    LANG = "pt"
    LANG_NAME = "Portuguese"
    DEEPL_TARGET = "PT-PT"   # use "PT-BR" if your references are Brazilian Portuguese
else:
    raise ValueError("PAIR must be 'en-es' or 'en-pt'")

print("Using", PAIR, "| Train:", len(train_ds), "Valid:", len(valid_ds), "| Target:", DEEPL_TARGET)


Using en-es | Train: 800 Valid: 200 | Target: ES


In [8]:
EQ_PATTERN   = re.compile(r"\b(?:log|ln|exp|sin|cos|tan)\s*\([^)]*\)")
LATEX_EQ     = re.compile(r"\$[^$]+\$")
CIT_PATTERN  = re.compile(r"[A-Z][a-z]+ et al\.\s*\(\d{4}\)")
PERC_PATTERN = re.compile(r"\d+(?:[.,]\d+)?\s*%")
NUM_PATTERN  = re.compile(r"\d+(?:[.,]\d+)?")

PROTECT_PATTERNS = [LATEX_EQ, EQ_PATTERN, CIT_PATTERN, PERC_PATTERN, NUM_PATTERN]

def rbmt_protect(text: str) -> Tuple[str, Dict[str, str]]:
    spans = []
    for pat in PROTECT_PATTERNS:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), m.group(0)))

    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    non_over, last_end = [], -1
    for s, e, v in spans:
        if s >= last_end:
            non_over.append((s, e, v))
            last_end = e

    out, cur, mp, k = [], 0, {}, 0
    for s, e, v in non_over:
        if cur < s:
            out.append(text[cur:s])
        ph = f"<ph{k}/>"
        mp[ph] = v
        out.append(ph)
        cur = e
        k += 1

    if cur < len(text):
        out.append(text[cur:])

    return "".join(out), mp

def rbmt_restore(text: str, mp: Dict[str, str]) -> str:
    for ph, val in mp.items():
        text = text.replace(ph, val)
    return text


In [9]:
def deepl_translate_one(text: str) -> str:
    protected, mp = rbmt_protect(text)

    result = translator.translate_text(
        protected,
        source_lang="EN",
        target_lang=DEEPL_TARGET,
        model_type="quality_optimized",
        preserve_formatting=True
    )

    out = result.text
    out = rbmt_restore(out, mp)
    return out


In [10]:
tests = [
    "We used a convolutional neural network for image classification.",
    "The dataset contains 800 training samples and 200 validation samples.",
    "Figure 2 shows the results of the ablation study.",
    "The patient was treated with antibiotics for 7 days.",
]

for i, src in enumerate(tests, 1):
    hyp = deepl_translate_one(src)
    print(f"\n[{i}] SRC: {src}\n    HYP: {hyp}")



[1] SRC: We used a convolutional neural network for image classification.
    HYP: Utilizamos una red neuronal convolucional para la clasificación de imágenes.

[2] SRC: The dataset contains 800 training samples and 200 validation samples.
    HYP: El conjunto de datos contiene 800 muestras de entrenamiento y 200 muestras de validación.

[3] SRC: Figure 2 shows the results of the ablation study.
    HYP: La figura 2 muestra los resultados del estudio de ablación.

[4] SRC: The patient was treated with antibiotics for 7 days.
    HYP: El paciente recibió tratamiento con antibióticos durante 7 días.


In [11]:
bleu_metric = evaluate.load("sacrebleu")
ter_metric = evaluate.load("ter")


In [12]:
def run_eval(valid_ds, pred_fn, max_n=None):
    n = len(valid_ds) if max_n is None else min(max_n, len(valid_ds))
    preds, refs = [], []

    for i in range(n):
        src = valid_ds[i]["src_text"]
        ref = valid_ds[i]["tgt_text"]
        hyp = pred_fn(src)

        preds.append(hyp)
        refs.append(ref)

        if (i + 1) % 20 == 0:
            print(f"Translated {i+1}/{n}")

    refs_ll = [[r] for r in refs]

    bleu = bleu_metric.compute(
        predictions=preds,
        references=refs_ll,
        use_effective_order=True
    )["score"]

    ter = ter_metric.compute(
        predictions=preds,
        references=refs_ll
    )["score"]

    exact = float(np.mean([p == r for p, r in zip(preds, refs)]) * 100.0)

    return {
        "n": n,
        "bleu": float(bleu),
        "ter": float(ter),
        "exact_match_%": exact,
        "predictions": preds,
        "references": refs,
    }


In [13]:
results = run_eval(valid_ds, pred_fn=deepl_translate_one, max_n=None)

print(f"{PAIR} DeepL -> BLEU {results['bleu']:.2f} | TER {results['ter']:.2f} | Exact {results['exact_match_%']:.2f}%")


Translated 20/200
Translated 40/200
Translated 60/200
Translated 80/200
Translated 100/200
Translated 120/200
Translated 140/200
Translated 160/200
Translated 180/200
Translated 200/200
en-es DeepL -> BLEU 32.79 | TER 54.74 | Exact 0.00%


In [14]:
for i in range(min(10, results["n"])):
    print("=" * 80)
    print("SRC :", valid_ds[i]["src_text"])
    print("REF :", results["references"][i])
    print("HYP :", results["predictions"][i])


SRC : Besides sublethal ischemia, other conditions, such as hyperthermia, hypothermia, hypoglycemia, and pharmacological agents e.g., antibiotics, erythropoietin, acetylsalicylic acid, and volatile anesthetics induce ischemic tolerance .
REF : Además de la isquemia subletal, otras condiciones tales como la hipertermia , hipotermia , hipoglucemia y los agentes farmacológicos, por ejemplo, antibióticos, eritropoetina, ácido acetilsalicílico y anestésicos volátiles, inducen a la tolerancia isquémica .
HYP : Además de la isquemia subletal, otras afecciones, como la hipertermia, la hipotermia y la hipoglucemia, así como determinados fármacos —por ejemplo, los antibióticos, la eritropoyetina, el ácido acetilsalicílico y los anestésicos volátiles— provocan tolerancia isquémica.
SRC : The open access movement initiated in the late 1990s aimed to eliminate economic and copyrights barriers in accessing and disseminating knowledge.
REF : El movimiento del acceso abierto, iniciado a finales de los